# Kontrole pomocnicze do dodatków A–D

Weryfikacja osi, masek, skalowania i kontroli pakietu. Wszystkie sygnały
w tym notebooku są kontrolne i syntetyczne. Argumentacja znajduje się
w tekstach dodatków. Uruchom z katalogu pakietu albo przez
`python scripts/run_appendices.py`. Brak pobierania danych i figur.

In [1]:
from pathlib import Path
import sys, json, platform, tempfile
import numpy as np
import scipy
from scipy import signal
from zipfile import ZipFile
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent]
            if (p / "scripts/audit_package.py").is_file())
sys.path.insert(0, str(ROOT / "scripts"))
from audit_package import inspect_archive
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
CHECKS = []
RESULTS = {}
def check(name, condition):
    if not bool(condition):
        raise AssertionError(name)
    CHECKS.append(name)
    print("OK:", name)
ENV = {"python": platform.python_version(), "numpy": np.__version__,
       "scipy": scipy.__version__, "execution": "ordinary Python cells"}
print(json.dumps(ENV, ensure_ascii=False))

{"python": "3.12.14", "numpy": "2.3.5", "scipy": "1.17.0", "execution": "ordinary Python cells"}


## A.1–A.3: osie, wycinki i maski

Kontrole rozróżniają czas lokalny i źródłowy oraz zachowanie luki
od skompaktowania próbek. Osobno sprawdzamy widok i kopię tablicy.

In [2]:
fs = 250.0
N = 1000
x = np.zeros(N, dtype=float)
t = np.arange(N) / fs
n0, n1 = 250, 750
fragment = x[n0:n1].copy()
time_record = np.arange(n0, n1) / fs
time_local = np.arange(n1 - n0) / fs
r_record = np.array([300, 500, 800])
r_local = r_record[(r_record >= n0) & (r_record < n1)] - n0
valid = np.ones(N, dtype=bool)
valid[100:125] = False
usable = valid & np.isfinite(x)
selected_values = x[usable]
selected_times = t[usable]
check("A: wycinek ma wyłączony koniec", len(fragment) == n1-n0)
check("A: początek osi jest odrębną informacją", np.allclose(time_record-time_local, n0/fs))
check("A: adnotacje lokalne", np.array_equal(r_local, [50, 250]))
check("A: maska zachowuje lukę na osi", np.isclose(np.diff(selected_times).max(), 26/fs))
check("A: kompresja próbek zmienia czas", not np.allclose(selected_times, np.arange(usable.sum())/fs))
y = np.arange(8.0)
view, independent = y[2:5], y[2:5].copy()
view[0] = 100
check("A: wycinek może zmienić źródło", y[2] == 100 and independent[0] == 2)
channels = np.column_stack((np.ones(8), 3*np.ones(8)))
check("A: oś agregacji rozróżnia kanały", np.array_equal(channels.mean(axis=0), [1, 3]) and channels.mean() == 2)
RESULTS["A"] = {"local_events": r_local.tolist(), "gap_s": float(np.diff(selected_times).max())}

OK: A: wycinek ma wyłączony koniec
OK: A: początek osi jest odrębną informacją
OK: A: adnotacje lokalne
OK: A: maska zachowuje lukę na osi
OK: A: kompresja próbek zmienia czas
OK: A: wycinek może zmienić źródło
OK: A: oś agregacji rozróżnia kanały


## A.4: kontrolowany plik NPZ

Fixture ma strukturę pól potrzebnych przykładowi A.4 i zawiera sygnał
syntetyczny; nie jest wycinkiem MIT-BIH. Przykład z tekstu sprawdzono
również redakcyjnie na rzeczywistym pliku rozdziału 1.

In [3]:
fixture = OUT / "demo_signal.npz"
np.savez(fixture, signal_mV=np.sin(2*np.pi*10*t), time_s=t,
         valid_mask=valid, metadata_json=json.dumps({"fs_hz": fs, "units": "mV"}))
with np.load(fixture, allow_pickle=False) as data:
    signal_mV = np.array(data["signal_mV"], dtype=float, copy=True)
    time_s = np.array(data["time_s"], dtype=float, copy=True)
    present = np.array(data["valid_mask"], dtype=bool, copy=True)
    metadata = json.loads(str(data["metadata_json"]))
fs_hz = float(metadata["fs_hz"])
check("A: geometria, jednostka i siatka NPZ", signal_mV.ndim == 1 and time_s.shape == present.shape == signal_mV.shape and metadata["units"] == "mV" and np.allclose(np.diff(time_s), 1/fs_hz))

OK: A: geometria, jednostka i siatka NPZ


## B: jednostronność, dopełnienie i jednostki

Porównanie z SciPy dla parzystych i nieparzystych długości DFT,
łącznie z DC i Nyquistem. Bilans jest liczony po pełnej siatce.

In [4]:
def manual_density(x, fs, w, nfft):
    if nfft < len(x):
        raise ValueError("Ta kontrola nie obcina wejścia.")
    z = np.fft.rfft(x*w, n=nfft)
    p = abs(z)**2 / (fs*np.sum(w*w))
    if nfft % 2 == 0:
        p[1:-1] *= 2
    else:
        p[1:] *= 2
    return np.fft.rfftfreq(nfft, 1/fs), p
rows = []
for N, nfft in [(255,255), (255,1024), (256,256), (256,1024)]:
    n = np.arange(N)
    x = .2 + .6*np.cos(2*np.pi*32*n/N)
    if N % 2 == 0:
        x = x + .1*(-1.)**n
    w = signal.windows.hann(N, sym=False)
    f, p = manual_density(x, 250., w, nfft)
    f_scipy, p_scipy = signal.periodogram(x, fs=250., window=w, detrend=False,
                  nfft=nfft, return_onesided=True, scaling="density")
    df = f[1]-f[0]
    area = float(p.sum()*df)
    target = float(np.sum((x*w)**2)/np.sum(w*w))
    check(f"B: zgodność z SciPy N={N} nfft={nfft}", np.allclose(f,f_scipy) and np.allclose(p,p_scipy,rtol=1e-12,atol=1e-14))
    check(f"B: bilans N={N} nfft={nfft}", np.isclose(area,target,rtol=1e-12,atol=1e-14))
    check(f"B: Nyquist zależy od długości DFT {nfft}", np.isclose(f[-1],125.) == (nfft%2 == 0))
    _, p_uv = manual_density(1000*x,250.,w,nfft)
    check(f"B: zmiana mV na uV N={N} nfft={nfft}", np.allclose(p_uv,1e6*p,rtol=1e-12,atol=1e-8))
    rows.append({"N":N,"nfft":nfft,"sum_psd_df":area,"weighted_mean_square":target})
RESULTS["B_density"] = rows

OK: B: zgodność z SciPy N=255 nfft=255
OK: B: bilans N=255 nfft=255
OK: B: Nyquist zależy od długości DFT 255
OK: B: zmiana mV na uV N=255 nfft=255
OK: B: zgodność z SciPy N=255 nfft=1024
OK: B: bilans N=255 nfft=1024
OK: B: Nyquist zależy od długości DFT 1024
OK: B: zmiana mV na uV N=255 nfft=1024
OK: B: zgodność z SciPy N=256 nfft=256
OK: B: bilans N=256 nfft=256
OK: B: Nyquist zależy od długości DFT 256
OK: B: zmiana mV na uV N=256 nfft=256
OK: B: zgodność z SciPy N=256 nfft=1024
OK: B: bilans N=256 nfft=1024
OK: B: Nyquist zależy od długości DFT 1024
OK: B: zmiana mV na uV N=256 nfft=1024


In [5]:
N, fs = 256, 250.0
n = np.arange(N)
w = signal.windows.hann(N, sym=False)
x = .2 + .6*np.cos(2*np.pi*32*n/N) + .1*(-1.)**n
amplitudes = []
for nfft in [N, 4*N]:
    a = abs(np.fft.rfft(x*w,nfft))/abs(w.sum())
    a[1:-1] *= 2
    k = 32*nfft//N
    check(f"B: amplitudy DC/ton/Nyquist nfft={nfft}", np.allclose([a[0],a[k],a[-1]],[.2,.6,.1],atol=1e-12))
    amplitudes.append([float(a[0]),float(a[k]),float(a[-1])])
f, p = manual_density(x, fs, w, N)
rectangle = float(p.sum()*(f[1]-f[0]))
trapezoid = float(np.trapezoid(p,f))
check("B: trapezy nie odtwarzają tego bilansu DC/Nyquista", not np.isclose(rectangle,trapezoid))
RESULTS["B_amplitude"] = amplitudes
RESULTS["B_integration"] = {"sum_df":rectangle,"trapezoid":trapezoid}

OK: B: amplitudy DC/ton/Nyquist nfft=256
OK: B: amplitudy DC/ton/Nyquist nfft=1024
OK: B: trapezy nie odtwarzają tego bilansu DC/Nyquista


## C: wynik dyskretny, czas i dostępność

Kontrola prostych definicji: identyczna liczba trafień nie określa
błędu czasu, a błąd HR przy braku porównywalnych okien jest nieokreślony.

In [6]:
ref = np.array([100, 300, 500])
estimate = ref + 10
fs = 250.
TP, FP, FN = 3, 0, 0
f1 = 2*TP/(2*TP+FP+FN)
mae_ms = float(np.mean(abs(estimate-ref))*1000/fs)
check("C: F1=1 może współistnieć z przesunięciem", f1 == 1 and mae_ms == 40.)
planned, available, comparable = 10, 2, 0
coverage = available/planned
mae_hr = None if comparable == 0 else 0.
check("C: brak porównania nie jest zerowym błędem", coverage == .2 and mae_hr is None)
RESULTS["C"] = {"F1": f1,"MAE_ms":mae_ms,"coverage":coverage,"MAE_HR":mae_hr}

OK: C: F1=1 może współistnieć z przesunięciem
OK: C: brak porównania nie jest zerowym błędem


## D: kompletne i ucięte archiwum

Tworzymy małe kontrolne archiwum. Druga kopia zachowuje dane wpisów,
ale traci katalog centralny. Osobno sprawdzamy brak wymaganego pliku.

In [7]:
with tempfile.TemporaryDirectory() as tmp:
    good = Path(tmp)/"good.zip"
    with ZipFile(good,"w") as z:
        z.writestr("demo/notebooks/example.ipynb","{}")
        z.writestr("demo/scripts/run.py","print('ok')")
    required = ["notebooks/example.ipynb","scripts/run.py"]
    full = inspect_archive(good,required)
    missing = inspect_archive(good,required+["src/helper.py"])
    raw = good.read_bytes()
    truncated = Path(tmp)/"truncated.zip"
    truncated.write_bytes(raw[:raw.index(b"PK\x01\x02")])
    broken = inspect_archive(truncated,required)
    check("D: poprawne archiwum i wspólny katalog", full["status"] == "passed")
    check("D: brak wymaganego kodu wykryty", missing["status"] == "failed")
    check("D: ucięcie wykryte mimo obecnych danych wpisów", broken["status"] == "failed")
RESULTS["D"] = {"complete":full["status"],"missing_file":missing["status"],"truncated":broken["status"]}

OK: D: poprawne archiwum i wspólny katalog
OK: D: brak wymaganego kodu wykryty
OK: D: ucięcie wykryte mimo obecnych danych wpisów


In [8]:
RESULTS["audit"] = {"passed_checks":len(CHECKS),"checks":CHECKS}
(OUT/"appendix_checks.json").write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
(OUT/"appendix_environment.json").write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print("Kontrole:",len(CHECKS))

Kontrole: 32
